In [ ]:
#| default_exp core

# SolveitCDP API
> Drive the user's Chrome from a solveit kernel

solvecdp is [fastcdp](https://github.com/AnswerDotAI/fastcdp) rebased onto solveit's infrastructure. A solveit kernel runs in a container on a remote server, so it can't open a websocket to the user's browser the way fastcdp does. Instead, frames travel over a `Channel` through rustygate's `/api/wsx` relay (exposed to browsers at `/rusty/api/wsx`) to the [solveit-chrome](https://github.com/AnswerDotAI/solveit-chrome) extension's service worker, which executes them with `chrome.debugger` and streams every CDP event back over the same channel.

`JsCDP` subclasses `fastcdp.ext`'s `ExtCDP`, which holds the whole extension frame protocol; the only thing added here is the channel, a `Channel` to the relay. Everything documented in fastcdp works unchanged: domain proxies with auto-generated signatures, `cdp_search`, the accessibility tree, form helpers, and the console/network/dialog debugging buffers. Chrome's integer `tabId`s play the role of fastcdp's session ids.

In [ ]:
from fastcore.utils import *
from fastcore.meta import delegates
import asyncio,concurrent.futures,json,uuid,websockets
from contextlib import suppress
from fastcdp.ext import *
import fastcdp.ext

In [ ]:
import threading
from fastcore.test import test_eq, expect_fail

`Channel` connects to rustygate's `/api/wsx` relay for ongoing JSON messaging. Set `JUPYGATE_URL` to the gateway's URL, or pass a full WebSocket `url` to `Channel.connect`. The default gateway is `http://127.0.0.1:8787`.

Rustygate supplies `JUPYGATE_URL` to every spawned kernel and trusts loopback connections; pass `token=` to send `Authorization: Bearer <token>` against a token-gated gateway. Peers join the same named channel even when the kernel connects directly over loopback and the browser or desktop app connects through Solveit's `/rusty/api/wsx` proxy.

In [ ]:
#| export
class Channel:
    "Duplex JSON messaging with the other peers on a named `/api/wsx` relay channel"
    def __init__(self, chan, url=None):
        base = os.environ.get('JUPYGATE_URL', 'http://127.0.0.1:8787').rstrip('/')
        self.url = url or f"{re.sub('^http', 'ws', base)}/api/wsx?chan={chan}"
        self._replies,self.events = {},asyncio.Queue()

    @classmethod
    async def connect(cls,
        chan, # Channel name; peers connecting to the same name hear each other
        url=None, # Full relay url (defaults to `/api/wsx` on `JUPYGATE_URL`)
        token=None # Optional bearer token for token-gated gateways
    ):
        "Connect to `chan` and start reading frames"
        self = cls(chan, url=url)
        headers = {'Authorization': f'Bearer {token}'} if token else None
        self._ws = await websockets.connect(self.url, additional_headers=headers)
        self._reader = asyncio.create_task(self._read_loop())
        return self

    async def _read_loop(self):
        async for frame in self._ws:
            msg = dict2obj(json.loads(frame))
            if (f := self._replies.pop(msg.get('id'), None)):
                if not f.done(): f.set_result(msg)
            else: await self.events.put(msg)

    async def send(self, **msg): await self._ws.send(json.dumps(msg))

    async def request(self, timeout=15, **msg):
        "Send `msg` with a unique `id` and wait for a reply with that `id`"
        mid = msg['id'] = str(uuid.uuid4())
        f = asyncio.get_running_loop().create_future()
        self._replies[mid] = f
        try:
            await self._ws.send(json.dumps(msg))
            return await asyncio.wait_for(f, timeout)
        finally: self._replies.pop(mid, None)

    async def close(self):
        self._reader.cancel()
        await self._ws.close()

    @property
    def is_open(self): return self._ws.state.name == 'OPEN'

Connect each peer to the same channel name on rustygate's `/api/wsx` relay. The relay forwards each frame unchanged to every other peer on that channel. `solvecdp` uses `Channel` to communicate with the [solveit-chrome](https://github.com/AnswerDotAI/solveit-chrome) extension.

`Channel` sends JSON dictionaries. Pass the fields as keyword arguments to `send`. Incoming frames become `AttrDict` objects. If a frame's `id` matches a pending request, it completes that request. Other frames go into the `events` queue.

Here we connect two peers and send a greeting. The second peer reads it from `events`.

In [ ]:
ch1,ch2 = await Channel.connect('test'),await Channel.connect('test')
await ch1.send(hello='world')
test_eq((await ch2.events.get()).hello, 'world')

`request` adds a unique `id` and waits for a reply with that `id`. Requests can run concurrently. The peer answering a request must include its `id` in the reply. This example replies with twice the supplied number.

In [ ]:
async def _responder():
    msg = await ch2.events.get()
    await ch2.send(id=msg.id, result=msg.x*2)

task = asyncio.create_task(_responder())
test_eq((await ch1.request(x=21)).result, 42)

Check `is_open` before and after closing the connections.

In [ ]:
assert ch1.is_open
await ch1.close()
await ch2.close()
assert not ch1.is_open

In [ ]:
class JsCDP(ExtCDP):
    "CDP over the solveit-chrome extension: frames relayed via rustygate's `/api/wsx` channel"
    @classmethod
    async def connect(cls,
        chan='cdp', # Relay channel name the extension serves
        debug=None # Print events as they arrive?
    ):
        "Connect to channel"
        return await super().connect(await Channel.connect(chan), debug=debug)

Everything else comes from `fastcdp.ext`'s `ExtCDP`: the `sessionId`↔`tabId` mapping, the lifecycle actions, and `new_page`/`attach_page`/`pages`. `JsCDP` just supplies the channel, meeting the extension at rustygate's `/api/wsx` relay instead of listening on a local port (a solveit kernel usually runs in a container that the user's browser can't reach, so both sides dial out to the relay).

Connect, open a tab, and drive it. `goto`, `eval`, and friends are all inherited from fastcdp through the `Page` proxy:


In [ ]:
cdp = await JsCDP.connect()
page = await cdp.new_page()
await page.goto('https://httpbingo.org/forms/post')
await page.eval('document.title')

In [ ]:
assert 'httpbingo.org' in await page.eval('document.title')
assert first(await cdp.pages, lambda t: 'httpbingo' in t.get('url',''))

The accessibility tree and form helpers also come straight from fastcdp: find elements with `ax_tree`/`find_id`, then `fill_text`, `click`, `select_option`, and `click_and_wait`.


In [ ]:
root = await page.ax_tree()
await page.fill_text(root.find_id('textbox', 'Customer name'), 'Jeremy Howard')
await page.click(root.find_id('radio', 'Large'))
await page.click(root.find_id('checkbox', 'Extra Cheese'))

In [ ]:
await page.click_and_wait(root.find_id('button', 'Submit order'))
assert 'Jeremy Howard' in await page.eval('document.body.innerText')

The debugging buffers work too, since every `chrome.debugger` event is streamed over the channel and dispatched by the inherited machinery:

In [ ]:
await page.start_console()
await page.eval(r'console.warn("watch out")')
test_eq(await page.console(r'watch'), ['warning: watch out'])

In [ ]:
await page.close()
await cdp.close()

## Running AppleScript with solvebridge

Because `/api/wsx` stores no messages and drops frames when no peer is connected, `_wait_bridge` waits until a `solvebridge` peer either replies `ok: true` to a `ping` probe or announces `{"event": "hello"}` on connect before sending an AppleScript command.

In [ ]:
async def _wait_hello(ch):
    while True:
        if (await ch.events.get()).get('event') == 'hello': return

async def _wait_bridge(ch, wait_connect=25, ping_interval=1.5):
    "Wait until a solvebridge peer answers `ping` or announces `hello` on `ch`"
    loop = asyncio.get_running_loop()
    deadline = loop.time() + wait_connect
    hello = asyncio.create_task(_wait_hello(ch))
    try:
        while (rem := deadline - loop.time()) > 0:
            ping = asyncio.create_task(ch.request(timeout=min(ping_interval, rem), cmd='ping'))
            done, _ = await asyncio.wait((hello, ping), return_when=asyncio.FIRST_COMPLETED)
            if hello in done:
                ping.cancel()
                with suppress(asyncio.CancelledError, TimeoutError): await ping
                return
            try:
                if (await ping).get('ok'): return
                await asyncio.sleep(min(ping_interval, max(0, deadline - loop.time())))
            except TimeoutError: pass
        raise TimeoutError(f"solvebridge is not connected on {ch.url}")
    finally:
        if not hello.done():
            hello.cancel()
            with suppress(asyncio.CancelledError): await hello

`aosa` opens a `Channel` (default `chan='mac'`), waits for `solvebridge` via `_wait_bridge`, and sends one `osascript` command. Replies carrying an `error` field (such as a user denial or unsupported platform) or a nonzero `returncode` raise `RuntimeError`.

In [ ]:
async def aosa(
    script:str, # AppleScript source to run on the connected Mac
    chan:str='mac', # Relay channel name `solvebridge` connects to
    url:str=None, # Full relay URL (defaults to `/api/wsx` on `JUPYGATE_URL`)
    token:str=None, # Optional bearer token for token-gated gateways
    timeout:float=60, # Seconds to wait for approval and execution
    wait_connect:float=25, # Seconds to wait for `solvebridge` to answer `ping` or `hello`
    ping_interval:float=1.5, # Seconds between `ping` probes while waiting for `solvebridge`
) -> str:
    "Run AppleScript `script` on the user's Mac through `solvebridge` and return stdout"
    ch = await Channel.connect(chan, url=url, token=token)
    try:
        await _wait_bridge(ch, wait_connect=wait_connect, ping_interval=ping_interval)
        res = await ch.request(timeout=timeout, cmd='osascript', script=script)
    finally: await ch.close()
    if (err := res.get('error')): raise RuntimeError(f"osascript: {err}")
    if (rc := res.get('returncode', 0)):
        msg = (res.get('stderr') or res.get('stdout') or '').strip()
        raise RuntimeError(f"osascript exited with status {rc}" + (f": {msg}" if msg else ''))
    return res.get('stdout', '')

Test the full handshake and error paths against a fake bridge peer on `/api/wsx`: late-connecting `hello` wakeup, `ping` replies, user denials, nonzero exit codes, disconnected timeouts, and calling synchronous `osa` while an event loop is already running.

In [ ]:
async def _fake_bridge(chan, delay=0, limit=None):
    if delay: await asyncio.sleep(delay)
    ch = await Channel.connect(chan)
    try:
        await ch.send(event='hello', platform='macos', app='solvebridge')
        seen = 0
        while limit is None or seen < limit:
            msg = await ch.events.get()
            if msg.get('cmd') == 'ping': await ch.send(id=msg.id, ok=True, platform='macos')
            elif msg.get('cmd') == 'osascript':
                seen += 1
                s = msg.script
                if s == 'deny': await ch.send(id=msg.id, error='denied by user')
                elif s == 'fail': await ch.send(id=msg.id, stdout='', stderr='syntax error', returncode=1)
                else: await ch.send(id=msg.id, stdout=f'ran: {s}\n', stderr='', returncode=0)
    finally: await ch.close()

In [ ]:
bridge = asyncio.create_task(_fake_bridge('test-mac', delay=0.08, limit=3))
test_eq(await aosa('activate', chan='test-mac', ping_interval=0.05), 'ran: activate\n')
with expect_fail(RuntimeError, 'denied by user'): await aosa('deny', chan='test-mac', ping_interval=0.05)
with expect_fail(RuntimeError, 'syntax error'): await aosa('fail', chan='test-mac', ping_interval=0.05)
await bridge
with expect_fail(TimeoutError, 'not connected'): await aosa('noop', chan='test-mac-empty', wait_connect=0.15, ping_interval=0.05)

`osa` wraps `aosa` synchronously so Solveit's `py` tool can run AppleScript without `await`, even inside a Jupyter kernel where an asyncio event loop is already running.

In [ ]:
def _run_sync(coro):
    try: asyncio.get_running_loop()
    except RuntimeError: return asyncio.run(coro)
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex: return ex.submit(asyncio.run, coro).result()

@delegates(aosa)
def osa(
    script:str, # AppleScript source to run on the connected Mac
    **kwargs
) -> str:
    "Run AppleScript `script` synchronously on the user's Mac through `solvebridge` and return stdout"
    return _run_sync(aosa(script, **kwargs))

In [ ]:
t = threading.Thread(target=asyncio.run, args=(_fake_bridge('test-mac-sync', delay=0.05, limit=1),))
t.start()
test_eq(osa('tell Word', chan='test-mac-sync', ping_interval=0.05), 'ran: tell Word\n')
t.join(2)
assert not t.is_alive()

In [ ]:
def cdp_yolo():
    "Allow all CDP classes and macOS bridge helpers in safepyrun"
    from pyskills import allow
    fastcdp.ext.cdp_yolo()
    allow({JsCDP:...}, osa, aosa)